In [ ]:
# ============================================
# JUPYTER NOTEBOOK - GraphRAG con Ollama + RDF
# ============================================

# ============================================
# CELDA 1 - IMPORTS
# ============================================

import os
import json
import re
import datetime

from time import time, sleep
from uuid import uuid4

from openai import OpenAI
from rdflib import Graph

# Imports propios
from searchInGraph import (
    buscar_frecuentes_por_opcion,
    inferir_valor_adecuado
)

from formatHelper import (
    extraer_support_category,
    extraer_cliente,
    formatear_para_llm,
    arreglar_lista_llm,
    #merge_listas_or,
    #limpiar_lista,
    aplicar_reglas
)

import config

In [ ]:
# ============================================
# CELDA 2 - CARGA DEL GRAFO RDF
# ============================================

graph = Graph()

graph.parse(
    config.TTL_FILE_PATH,
    format=config.TTL_FORMAT
)

print("Grafo cargado correctamente")
print(f"Número de triples: {len(graph)}")

In [ ]:
# ============================================
# CELDA 3 - CONFIGURACIÓN DEL MODELO
# ============================================

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

mi_modelo = "mistral:latest"

print(f"Modelo configurado: {mi_modelo}")

In [ ]:
# ============================================
# CELDA 4 - FUNCIONES AUXILIARES
# ============================================

def open_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as infile:
        return infile.read()


def save_file(filepath, content):
    with open(filepath, 'w', encoding='utf-8') as outfile:
        outfile.write(content)


def load_json(filepath):
    with open(filepath, 'r', encoding='utf-8') as infile:
        return json.load(infile)


def save_json(filepath, payload):
    with open(filepath, 'w', encoding='utf-8') as outfile:
        json.dump(
            payload,
            outfile,
            ensure_ascii=False,
            sort_keys=True,
            indent=2
        )


def timestamp_to_datetime(unix_time):
    return datetime.datetime.fromtimestamp(
        unix_time
    ).strftime("%A, %B %d, %Y at %I:%M%p %Z")

In [ ]:
# ============================================
# CELDA 5 - FUNCIÓN DE COMPLETADO LLM
# ============================================

def text_completion(prompt, engine=config.MI_MODELO):

    max_retry = 5
    retry = 0

    while True:

        try:

            response = client.chat.completions.create(
                messages=[
                    {
                        "role": "user",
                        "content": prompt,
                    }
                ],
                model=engine
            )

            text = response.choices[0].message.content

            # Limpieza básica
            text = re.sub(r'[\r\n]+', '\n', text)
            text = re.sub(r'[\t ]+', ' ', text)

            return text

        except Exception as oops:

            retry += 1

            if retry >= max_retry:
                return f"Model error: {oops}"

            print("Error comunicando con el modelo:", oops)

            sleep(config.RETRY_DELAY_SECONDS)

In [ ]:
# ============================================
# CELDA 6 - VARIABLES DE ESTADO
# ============================================

convo_length = 2

unique_conv_id = str(uuid4())

prev_conv = ""

filename = unique_conv_id + "_log.txt"

log_file_path = os.path.join(
    config.LOGS_DIR,
    filename
)

save_file(log_file_path, prev_conv)

primera = True

buscar = False

mi_opcion = None

cat_buscar = 0

graph_data = []

# Estructura:
# 0 - Int_hasCustomer
# 1 - hasSupportCategory
# 2 - hasTypeInc
# 3 - incident_hasOrigin
# 4 - hasSupportGroup
# 5 - hasTechnician

mis_datos = [None, None, None, None, None, None]

print("Sistema inicializado")
print("Estado actual:", mis_datos)

In [ ]:
# ============================================
# CELDA 7 - FUNCIÓN PRINCIPAL DEL CHAT (AUTO-OPCIÓN 1)
# ============================================
contadores = {"vecesdf": 0, "vecesnv": 0}

vecesretry = 0


def procesar_mensaje_usuario(a):

    global primera
    global mis_datos
    global graph_data
    global cat_buscar
    global mi_opcion
    global prev_conv
    mis_datos = [None, None, None, None, None, None]
    # Normalizamos la entrada del usuario para validación de salida
    a_clean = str(a).strip().lower()

    if a_clean == "q":
        print("Finalizando conversación")
        return False

    # ========================================
    # GUARDAR MENSAJE RECIBIDO
    # ========================================
    timestamp = time()
    timestring = timestamp_to_datetime(timestamp)
    message = f"USER: {timestring} - {a}"

    buscar = True

    # ========================================
    # EVALUAR RESPUESTA DEL MENÚ (SIEMPRE ASUME 1)
    # ========================================
    if graph_data:
        # Se elimina la lógica de "sí", "no" y múltiples números.
        # Siempre se toma la primera opción devuelta por GraphRAG.
        mis_datos[cat_buscar] = graph_data[0]
        #print(f"\nGraphRAG: Asignado automáticamente '{graph_data[0]}' (Opción 1) a la categoría {cat_buscar}")
        
        # Reiniciamos las opciones para buscar la siguiente categoría faltante
        graph_data = [] 

    # ========================================
    # EXTRACCIÓN DE DATOS INICIALES (0 y 1)
    # ========================================
    if mis_datos[0] is None or mis_datos[0] == 'None':
        cliente = extraer_cliente(a)
        if cliente is not None:
            mis_datos[0] = cliente

    if mis_datos[1] is None or mis_datos[1] == 'None':
        support_cat = extraer_support_category(a)
        if support_cat is not None:
            mis_datos[1] = support_cat

    # ========================================
    # VERIFICAR SI YA ESTÁ COMPLETO
    # ========================================
    if None not in mis_datos and 'None' not in mis_datos:
        print(f'\nGraphRAG: query acabada. La query es {mis_datos}')
        return False

    # ========================================
    # BUSCAR CATEGORÍA FALTANTE (SI CORRESPONDE)
    # ========================================
    if buscar:
        try:
            cat_buscar = mis_datos.index(None)
        except ValueError:
            cat_buscar = mis_datos.index('None')

        graph_data = buscar_frecuentes_por_opcion(graph, mis_datos, cat_buscar)

        if not graph_data:
            graph_data = inferir_valor_adecuado(graph, mis_datos, cat_buscar)
    
    
        if graph_data:
                graph_data = aplicar_reglas(
                    './textos/reglas_incidentes.json', # Ajusta la ruta a tu fichero
                    mis_datos, 
                    cat_buscar, 
                    graph_data, contadores
                )
                
                if graph_data == []:
                    graph_data = inferir_valor_adecuado(graph, mis_datos, cat_buscar)
                    graph_data = aplicar_reglas(
                    './textos/reglas_incidentes.json', # Ajusta la ruta a tu fichero
                    mis_datos, 
                    cat_buscar, 
                    graph_data, contadores
                )
    
    
    if graph_data:
        mi_opcion = graph_data[0]

    # ========================================
    # PREPARACIÓN DE RESPUESTA EN LOGS / CONSOLA
    # ========================================
    prev_conv = open_file(log_file_path)

    if not graph_data:
        output = "No se encontraron datos. Seguramente sea un error por parte del usuario. Pregunta si se ha introducido bien el grupo."
        print(f"\n[Asistente] {output}")
    else:
        # Solo mostramos la Opción 1 ya que es la única que tomará el sistema
        output = f"¿Cuál es el valor para {config.DICCIONARIO_PREFIJOS[cat_buscar]}?\n 1. {mi_opcion}\n(Presiona Enter para continuar, se asignará esta opción automáticamente)"
        #print(f"\n[Asistente] {output}")

    # ========================================
    # GUARDAR CONVERSACIÓN
    # ========================================
    messageBot = f"[Asistente]: {timestring} - {output}"

    save_file(
        log_file_path,
        prev_conv + "\n" + message + "\n" + messageBot
    )

    return True, mis_datos





In [ ]:
## ============================================
## CELDA 8 - BUCLE INTERACTIVO
## ============================================
#
#print("====================================")
#print(" SISTEMA GraphRAG + Ollama INICIADO")
#print("====================================")
#print("Escribe 'q' para salir")
#print()
#
#primero = True
#
#while True:
#    
#    if primero:
#        entrada = input("USER: ")
#    
#    primero = False
#    
#    continuar, mis_datos = procesar_mensaje_usuario(entrada)
#    
#    
#    
#    
#    if not continuar:
#        break
#
#

In [ ]:
def testear_grafo(g, prefix_uri="http://repcon.org/schema#"):
    """
    Función de diagnóstico para comprobar el estado del grafo y 
    probar las funciones de búsqueda e inferencia de incidentes.
    """
    print("\n" + "="*50)
    print(" INICIANDO TEST DEL GRAFO ".center(50, "="))
    print("="*50)

    # ========================================
    # 1. TAMAÑO DEL GRAFO
    # ========================================
    print("\n[1] Comprobando tamaño del grafo...")
    try:
        print(f"Total de tripletas cargadas: {len(g)}")
    except Exception as e:
        print(f"Error al leer la longitud del grafo: {e}")

   # ========================================
    # 2. TEST DE CONTENIDO BÁSICO (Top 5 Clientes)
    # ========================================
    print("\n[2] Obteniendo los 5 clientes más frecuentes (int_hasCustomer)...")
    query_basica = f"""
    SELECT ?cliente (COUNT(?cliente) AS ?total)
    WHERE {{
        ?incident <{prefix_uri}int_hasCustomer> ?cliente .
    }}
    GROUP BY ?cliente
    ORDER BY DESC(?total)
    LIMIT 5
    """
    
    try:
        # Convertimos a lista para evitar problemas con el generador de rdflib
        resultados = list(g.query(query_basica))
        
        if not resultados:
            print(f"  ⚠ No hay datos para el predicado '{prefix_uri}int_hasCustomer'.")
            print("  🔍 Inspeccionando los 5 predicados que MÁS se repiten en tu grafo...")
            
            query_rescate = """
            SELECT ?p (COUNT(?p) AS ?total)
            WHERE { ?s ?p ?o . }
            GROUP BY ?p
            ORDER BY DESC(?total)
            LIMIT 5
            """
            resultados_rescate = g.query(query_rescate)
            for r in resultados_rescate:
                print(f"    - {r.p} (Apariciones: {r.total})")
        else:
            for row in resultados:
                val = str(row.cliente).split("#")[-1] if "#" in str(row.cliente) else str(row.cliente).rsplit("/", 1)[-1]
                print(f"  - {val} (Apariciones: {row.total})")
                
    except Exception as e:
        print(f"  Error en consulta básica: {e}")

    # ========================================
    # 3. TEST DE TUS FUNCIONES
    # ========================================
    print("\n[3] Probando tus funciones de filtrado e inferencia...")
    
    # Simulamos el array 'mis_datos' (6 posiciones)
    # Suponemos que ya tenemos el cliente (índice 0), y queremos buscar la Categoría (índice 1)
    # Índices: [Customer, SupportCategory, TypeInc, Origin, SupportGroup, Technician]
    
    # ⚠️ IMPORTANTE: Cambia "Cliente_Prueba" por el nombre de un cliente real de tu grafo para testear
    datos_simulados = ["Cliente_Prueba", None, None, None, None, None]
    categoria_a_buscar = 1  # 1 = hasSupportCategory

    print(f"  Estado simulado (mis_datos): {datos_simulados}")
    print(f"  Índice a buscar: {categoria_a_buscar} (hasSupportCategory)")

    # 3.1 Test: buscar_frecuentes_por_opcion
    print("\n  >>> Ejecutando 'buscar_frecuentes_por_opcion'...")
    try:
        res_busqueda = buscar_frecuentes_por_opcion(g, datos_simulados, categoria_a_buscar, prefix_uri)
        print(f"  Resultado Búsqueda Exacta: {res_busqueda}")
    except Exception as e:
        print(f"  Error en buscar_frecuentes_por_opcion: {e}")

    # 3.2 Test: inferir_valor_adecuado
    print("\n  >>> Ejecutando 'inferir_valor_adecuado' (Fallback)...")
    try:
        res_inferencia = inferir_valor_adecuado(g, datos_simulados, categoria_a_buscar, prefix_uri)
        print(f"  Resultado Inferencia: {res_inferencia}")
    except Exception as e:
        print(f"  Error en inferir_valor_adecuado: {e}")

    print("\n" + "="*50)
    print(" TEST FINALIZADO ".center(50, "="))
    print("="*50 + "\n")

In [ ]:
# Asumiendo que tu grafo se llama 'graph' en el entorno global:
# ⚠️ Cambia el "Cliente_Prueba" en la función por un string que sepas que sí existe en tu ontología
testear_grafo(graph)

In [ ]:
# ============================================
# CELDA 9 - VISUALIZAR ESTADO FINAL
# ============================================

print("====================================")
print(" ESTADO FINAL")
print("====================================")

labels = [
    "Cliente",
    "SupportCategory",
    "TypeInc",
    "Origin",
    "SupportGroup",
    "Technician"
]

for i, valor in enumerate(mis_datos):

    print(f"{labels[i]} -> {valor}")

In [ ]:
# ============================================
# CELDA 10 - VISUALIZAR LOG
# ============================================

print("====================================")
print(" LOG DE CONVERSACIÓN")
print("====================================")

contenido_log = open_file(log_file_path)

print(contenido_log)

In [ ]:
import unittest

mis_datos = [None, None, None, None, None, None]

def procesar_query(texto):
    
    veces = 0
    primero = True

    while True:

        if primero:
            entrada = texto

        primero = False

        continuar = procesar_mensaje_usuario(entrada)
        
        if veces >= 10:
            continuar = False
        
        veces = veces +1
        if not continuar:
            break
    
    # Aquí iría tu código real (regex, NLP, etc.)
    # Devuelvo una lista vacía solo para que el código no dé error de compilación
    return mis_datos

class TestQueryProcessor(unittest.TestCase):
    def setUp(self):
        self.casos_de_prueba = [

            # ─────────────────────────────────────────────────────────────────
            # BLOQUE 1 · Sin reglas  (casos 1–10)
            # Ninguna condición de las reglas df ni noValid se cumple.
            # ─────────────────────────────────────────────────────────────────

            # Caso 1 · incident_14976100001762302666
            (
                "Hola quiero completar una query. Tengo el supportCategory_149762111762302662 y la empresa company__QCTRKWQRI",
                ['company__QCTRKWQRI', 'supportCategory_149762111762302662',
                 'typeIncident__2', 'incidentOrigin__2',
                 'supportGroup_149762121762302662', None]
            ),

            # Caso 2 · incident_1497610000191762303983
            (
                "Hola quiero completar una query. Tengo el supportCategory_149768521762302662 y la empresa company__OFO40S6U5",
                ['company__OFO40S6U5', 'supportCategory_149768521762302662',
                 'typeIncident__2', 'incidentOrigin__2',
                 'supportGroup_149761521762302662', 'employee__403']
            ),

            # Caso 3 · incident_1497610000341762303983
            (
                "Hola quiero completar una query. Tengo el supportCategory_149761991762302662 y la empresa company__WOQ4SSP19",
                ['company__WOQ4SSP19', 'supportCategory_149761991762302662',
                 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_1497684871762302665', None]
            ),

            # Caso 4 · incident_1497610000351762303983
            (
                "Hola quiero completar una query. Tengo el supportCategory_149768521762302662 y la empresa company__2F48IOMAX",
                ['company__2F48IOMAX', 'supportCategory_149768521762302662',
                 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149761521762302662', 'employee__39']
            ),

            # Caso 5 · incident_1497610000481762303983
            (
                "Hola quiero completar una query. Tengo el supportCategory_1497610291762302662 y la empresa company__CFD5UKZBE",
                ['company__CFD5UKZBE', 'supportCategory_1497610291762302662',
                 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_1497610301762302662', 'employee__239']
            ),

            # Caso 6 · incident_1497610000991762303983
            (
                "Hola quiero completar una query. Tengo el supportCategory_149767291231762303563 y la empresa company__1GR6455ID",
                ['company__1GR6455ID', 'supportCategory_149767291231762303563',
                 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149766077241762303391', 'employee__259']
            ),

            # Caso 7 · incident_1497610002031762303983
            (
                "Hola quiero completar una query. Tengo el supportCategory_1497611711762302662 y la empresa company__ERAWDJ1I6",
                ['company__ERAWDJ1I6', 'supportCategory_1497611711762302662',
                 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149764431762302662', None]
            ),

            # Caso 8 · incident_1497610002231762303983
            (
                "Hola quiero completar una query. Tengo el supportCategory_149768010131762303673 y la empresa company__1GR6455ID",
                ['company__1GR6455ID', 'supportCategory_149768010131762303673',
                 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149766077241762303391', 'employee__170']
            ),

            # Caso 9 · incident_1497610002701762303983
            (
                "Hola quiero completar una query. Tengo el supportCategory_149766041762302662 y la empresa company__A-20042693",
                ['company__A-20042693', 'supportCategory_149766041762302662',
                 'typeIncident__2', 'incidentOrigin__2',
                 'supportGroup_149761634961762302818', 'employee__299']
            ),

            # Caso 10 · incident_1497610002911762303983
            (
                "Hola quiero completar una query. Tengo el supportCategory_1497624411762302663 y la empresa company__6RLLP0TOA",
                ['company__6RLLP0TOA', 'supportCategory_1497624411762302663',
                 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149761521762302662', None]
            ),

            # ─────────────────────────────────────────────────────────────────
            # BLOQUE 2 · Solo reglas df  (casos 11–20)
            # Al menos una regla df se activa; ninguna noValid aplica.
            # ─────────────────────────────────────────────────────────────────

            # Caso 11 · incident_1497610000031762303983
            # DF: incident_hasOrigin=incidentOrigin__3 → hasSupportGroup=supportGroup_149762881762302662
            (
                "Hola quiero completar una query. Tengo el supportCategory_149766571762302662 y la empresa company__9G1G3MV0P",
                ['company__9G1G3MV0P', 'supportCategory_149766571762302662',
                 'typeIncident__1', 'incidentOrigin__3',
                 'supportGroup_149762611762302662', 'employee__23']
            ),

            # Caso 12 · incident_1497610000551762303983
            # DF: incident_hasOrigin=incidentOrigin__3 → hasSupportGroup=supportGroup_149762881762302662
            (
                "Hola quiero completar una query. Tengo el supportCategory_149761881762302662 y la empresa company_149761700091762302830",
                ['company_149761700091762302830', 'supportCategory_149761881762302662',
                 'typeIncident__2', 'incidentOrigin__3',
                 'supportGroup_1497611281762302662', 'employee__294']
            ),

            # Caso 13 · incident_1497610000651762303983
            # DF: incident_hasOrigin=incidentOrigin__3 → hasSupportGroup=supportGroup_149762881762302662
            (
                "Hola quiero completar una query. Tengo el supportCategory_149769391762302662 y la empresa company__D4SFXFY9YR6",
                ['company__D4SFXFY9YR6', 'supportCategory_149769391762302662',
                 'typeIncident__2', 'incidentOrigin__3',
                 'supportGroup_149762611762302662', 'employee__601']
            ),

            # Caso 14 · incident_1497610001201762303983
            # DF: incident_hasOrigin=incidentOrigin__3 → hasSupportGroup=supportGroup_149762881762302662
            (
                "Hola quiero completar una query. Tengo el supportCategory_1497633831762302663 y la empresa company__PJ42MKUE7",
                ['company__PJ42MKUE7', 'supportCategory_1497633831762302663',
                 'typeIncident__1', 'incidentOrigin__3',
                 'supportGroup_149761661762302662', 'employee__437']
            ),

            # Caso 15 · incident_149761000151762302746
            # DF: incident_hasOrigin=incidentOrigin__3 → hasSupportGroup=supportGroup_149762881762302662
            (
                "Hola quiero completar una query. Tengo el supportCategory_1497640311762302663 y la empresa company__7POLMU91Q",
                ['company__7POLMU91Q', 'supportCategory_1497640311762302663',
                 'typeIncident__2', 'incidentOrigin__3',
                 'supportGroup_149761661762302662', 'employee__147']
            ),

            # Caso 16 · incident_1497610001721762303983
            # DF: incident_hasOrigin=incidentOrigin__3 → hasSupportGroup=supportGroup_149762881762302662
            (
                "Hola quiero completar una query. Tengo el supportCategory_149763371762302662 y la empresa company_14976254351762302678",
                ['company_14976254351762302678', 'supportCategory_149763371762302662',
                 'typeIncident__1', 'incidentOrigin__3',
                 'supportGroup_149761461762302662', 'employee__499']
            ),

            # Caso 17 · incident_1497610001921762303983
            # DF: incident_hasOrigin=incidentOrigin__3 → hasSupportGroup=supportGroup_149762881762302662
            (
                "Hola quiero completar una query. Tengo el supportCategory_149763371762302662 y la empresa company__F7UMNAXNO",
                ['company__F7UMNAXNO', 'supportCategory_149763371762302662',
                 'typeIncident__1', 'incidentOrigin__3',
                 'supportGroup_149761461762302662', 'employee__301']
            ),

            # Caso 18 · incident_14976100021762302666
            # DF: incident_hasOrigin=incidentOrigin__3 → hasSupportGroup=supportGroup_149762881762302662
            (
                "Hola quiero completar una query. Tengo el supportCategory_149761451762302662 y la empresa company__9G1G3MV0P",
                ['company__9G1G3MV0P', 'supportCategory_149761451762302662',
                 'typeIncident__1', 'incidentOrigin__3',
                 'supportGroup_149761461762302662', 'employee__23']
            ),

            # Caso 19 · incident_14976100021762304093
            # DF: incident_hasOrigin=incidentOrigin__3 → hasSupportGroup=supportGroup_149762881762302662
            (
                "Hola quiero completar una query. Tengo el supportCategory_1497658141762302664 y la empresa company_149763138681762303002",
                ['company_149763138681762303002', 'supportCategory_1497658141762302664',
                 'typeIncident__2', 'incidentOrigin__3',
                 'supportGroup_149762611762302662', None]
            ),

            # Caso 20 · incident_149761000241762302746
            # DF: incident_hasOrigin=incidentOrigin__3 → hasSupportGroup=supportGroup_149762881762302662
            (
                "Hola quiero completar una query. Tengo el supportCategory_149766571762302662 y la empresa company__U4S8LL2ZS",
                ['company__U4S8LL2ZS', 'supportCategory_149766571762302662',
                 'typeIncident__2', 'incidentOrigin__3',
                 'supportGroup_149762611762302662', 'employee__297']
            ),

            # ─────────────────────────────────────────────────────────────────
            # BLOQUE 3 · Solo reglas noValid  (casos 21–30)
            # Al menos una regla noValid se activa; ninguna df aplica.
            # ─────────────────────────────────────────────────────────────────

            # Caso 21 · incident_1497610000841762303983
            # NV: incident_hasOrigin=incidentOrigin__1 → hasTypeInc≠typeIncident__1
            (
                "Hola quiero completar una query. Tengo el supportCategory_149764053791762303121 y la empresa company__SWQM2V2KX",
                ['company__SWQM2V2KX', 'supportCategory_149764053791762303121',
                 'typeIncident__1', 'incidentOrigin__1',
                 'supportGroup_149761521762302662', 'employee__482']
            ),

            # Caso 22 · incident_1497610000881762303983
            # NV: incident_hasOrigin=incidentOrigin__1 → hasTypeInc≠typeIncident__1
            (
                "Hola quiero completar una query. Tengo el supportCategory_149764053791762303121 y la empresa company__SWQM2V2KX",
                ['company__SWQM2V2KX', 'supportCategory_149764053791762303121',
                 'typeIncident__1', 'incidentOrigin__1',
                 'supportGroup_149761521762302662', 'employee__482']
            ),

            # Caso 23 · incident_1497610003341762303983
            # NV: incident_hasOrigin=incidentOrigin__1 → hasTypeInc≠typeIncident__1
            (
                "Hola quiero completar una query. Tengo el supportCategory_149761511762302662 y la empresa company__UW9JVILER",
                ['company__UW9JVILER', 'supportCategory_149761511762302662',
                 'typeIncident__2', 'incidentOrigin__1',
                 'supportGroup_149761521762302662', 'employee__39']
            ),

            # Caso 24 · incident_1497610003931762303984
            # NV: hasSupportTeam=supportTeam_149762304211762302899
            #     → hasSupportCategory≠supportCategory_149763514921762303044
            (
                "Hola quiero completar una query. Tengo el supportCategory_14976149461762302668 y la empresa company__CFD5UKZBE",
                ['company__CFD5UKZBE', 'supportCategory_14976149461762302668',
                 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_1497611281762302662', 'employee__432']
            ),

            # Caso 25 · incident_1497610006081762303984
            # NV: incident_hasOrigin=incidentOrigin__1 → hasTypeInc≠typeIncident__1
            (
                "Hola quiero completar una query. Tengo el supportCategory_14976605081762302710 y la empresa company_149762729281762302954",
                ['company_149762729281762302954', 'supportCategory_14976605081762302710',
                 'typeIncident__1', 'incidentOrigin__1',
                 'supportGroup_149761521762302662', 'employee__357']
            ),

            # Caso 26 · incident_1497610008531762303984
            # NV: hasSupportTeam=supportTeam_149762304211762302899
            #     → hasSupportCategory≠supportCategory_149763514921762303044
            (
                "Hola quiero completar una query. Tengo el supportCategory_149761111762302662 y la empresa company__CFD5UKZBE",
                ['company__CFD5UKZBE', 'supportCategory_149761111762302662',
                 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149761121762302662', 'employee__432']
            ),

            # Caso 27 · incident_1497610008791762303984
            # NV: incident_hasOrigin=incidentOrigin__1 → hasTypeInc≠typeIncident__1
            (
                "Hola quiero completar una query. Tengo el supportCategory_149768051762302662 y la empresa company__7KWU1RCLX2",
                ['company__7KWU1RCLX2', 'supportCategory_149768051762302662',
                 'typeIncident__1', 'incidentOrigin__1',
                 'supportGroup_149762921762302662', None]
            ),

            # Caso 28 · incident_1497610009011762303984
            # NV: incident_hasOrigin=incidentOrigin__1 → hasTypeInc≠typeIncident__1
            (
                "Hola quiero completar una query. Tengo el supportCategory_149763128321762303001 y la empresa company__7KWU1RCLX2",
                ['company__7KWU1RCLX2', 'supportCategory_149763128321762303001',
                 'typeIncident__2', 'incidentOrigin__1',
                 'supportGroup_149762761762302662', None]
            ),

            # Caso 29 · incident_1497610009021762303984
            # NV: incident_hasOrigin=incidentOrigin__1 → hasTypeInc≠typeIncident__1
            (
                "Hola quiero completar una query. Tengo el supportCategory_149761591762302662 y la empresa company__7KWU1RCLX2",
                ['company__7KWU1RCLX2', 'supportCategory_149761591762302662',
                 'typeIncident__1', 'incidentOrigin__1',
                 'supportGroup_149762921762302662', None]
            ),

            # Caso 30 · incident_1497610009611762303984
            # NV: hasSupportTeam=supportTeam_149762304211762302899
            #     → hasSupportCategory≠supportCategory_149763514921762303044
            (
                "Hola quiero completar una query. Tengo el supportCategory_149761111762302662 y la empresa company__CFD5UKZBE",
                ['company__CFD5UKZBE', 'supportCategory_149761111762302662',
                 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149761121762302662', 'employee__430']
            ),

            # ─────────────────────────────────────────────────────────────────
            # BLOQUE 4 · Ambas reglas (df y noValid) sin contradicción  (casos 31–40)
            # Reglas df y noValid se activan, pero sobre predicados distintos.
            # ─────────────────────────────────────────────────────────────────

            # Caso 31 · incident_1497610000081762303983
            # DF: int_hasCustomer=ss → hasStateIncident=statusIncident__2
            # DF: int_hasCustomer=ss + hasTypeInc=typeIncident__1
            #     → hasSupportGroup=supportGroup_14976691762302662
            # NV: int_hasCustomer=ss → incident_hasOrigin≠incidentOrigin__1
            (
                "Hola quiero completar una query. Tengo el supportCategory_14976256531762302678 y la empresa ss",
                ['ss', 'supportCategory_14976256531762302678',
                 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976212821762302676', 'employee__439']
            ),

            # Caso 32 · incident_1497610000101762303983
            # DF: incident_hasOrigin=incidentOrigin__3 → hasSupportGroup=supportGroup_149762881762302662
            # NV: hasSupportTeam=supportTeam_149762304211762302899
            #     → hasSupportCategory≠supportCategory_149763514921762303044
            (
                "Hola quiero completar una query. Tengo el supportCategory_149762111762302662 y la empresa company__9G1G3MV0P",
                ['company__9G1G3MV0P', 'supportCategory_149762111762302662',
                 'typeIncident__2', 'incidentOrigin__3',
                 'supportGroup_149762611762302662', 'employee__487']
            ),

            # Caso 33 · incident_1497610000281762303983
            # DF: int_hasCustomer=ss → hasStateIncident=statusIncident__2
            # DF: int_hasCustomer=ss + hasTypeInc=typeIncident__1
            #     → hasSupportGroup=supportGroup_14976691762302662
            # NV: int_hasCustomer=ss → incident_hasOrigin≠incidentOrigin__1
            (
                "Hola quiero completar una query. Tengo el supportCategory_1497673211762302665 y la empresa ss",
                ['ss', 'supportCategory_1497673211762302665',
                 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149761521762302662', 'employee__403']
            ),

            # Caso 34 · incident_1497610000511762303983
            # DF: int_hasCustomer=ss → hasStateIncident=statusIncident__2
            # DF: int_hasCustomer=ss + hasTypeInc=typeIncident__1
            #     → hasSupportGroup=supportGroup_14976691762302662
            # NV: int_hasCustomer=ss → incident_hasOrigin≠incidentOrigin__1
            (
                "Hola quiero completar una query. Tengo el supportCategory_1497644851762302663 y la empresa ss",
                ['ss', 'supportCategory_1497644851762302663',
                 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149761661762302662', 'employee__442']
            ),

            # Caso 35 · incident_149761000061762302746
            # DF: incident_hasOrigin=incidentOrigin__3 → hasSupportGroup=supportGroup_149762881762302662
            # NV: hasTechnician=employee__365 → incident_hasOrigin≠incidentOrigin__4
            (
                "Hola quiero completar una query. Tengo el supportCategory_1497657281762302664 y la empresa company__8K9GJEIFB",
                ['company__8K9GJEIFB', 'supportCategory_1497657281762302664',
                 'typeIncident__2', 'incidentOrigin__3',
                 'supportGroup_149762881762302662', 'employee__365']
            ),

            # Caso 36 · incident_1497610000681762303983
            # DF: int_hasCustomer=ss → hasStateIncident=statusIncident__2
            # DF: int_hasCustomer=ss + hasTypeInc=typeIncident__1
            #     → hasSupportGroup=supportGroup_14976691762302662
            # NV: int_hasCustomer=ss → incident_hasOrigin≠incidentOrigin__1
            (
                "Hola quiero completar una query. Tengo el supportCategory_14976212811762302676 y la empresa ss",
                ['ss', 'supportCategory_14976212811762302676',
                 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976212821762302676', 'employee__108']
            ),

            # Caso 37 · incident_1497610000971762303983
            # DF: int_hasCustomer=ss → hasStateIncident=statusIncident__2
            # DF: int_hasCustomer=ss + hasTypeInc=typeIncident__1
            #     → hasSupportGroup=supportGroup_14976691762302662
            # NV: int_hasCustomer=ss → incident_hasOrigin≠incidentOrigin__1
            (
                "Hola quiero completar una query. Tengo el supportCategory_14976212811762302676 y la empresa ss",
                ['ss', 'supportCategory_14976212811762302676',
                 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976212821762302676', None]
            ),

            # Caso 38 · incident_1497610001141762303983
            # DF: int_hasCustomer=ss → hasStateIncident=statusIncident__2
            # DF: int_hasCustomer=ss + hasTypeInc=typeIncident__1
            #     → hasSupportGroup=supportGroup_14976691762302662
            # NV: int_hasCustomer=ss → incident_hasOrigin≠incidentOrigin__1
            (
                "Hola quiero completar una query. Tengo el supportCategory_14976212811762302676 y la empresa ss",
                ['ss', 'supportCategory_14976212811762302676',
                 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976212821762302676', 'employee__171']
            ),

            # Caso 39 · incident_1497610001751762303983
            # DF: int_hasCustomer=ss → hasStateIncident=statusIncident__2
            # DF: int_hasCustomer=ss + hasTypeInc=typeIncident__1
            #     → hasSupportGroup=supportGroup_14976691762302662
            # NV: int_hasCustomer=ss → incident_hasOrigin≠incidentOrigin__1
            (
                "Hola quiero completar una query. Tengo el supportCategory_14976212811762302676 y la empresa ss",
                ['ss', 'supportCategory_14976212811762302676',
                 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976212821762302676', 'employee__241']
            ),

            # Caso 40 · incident_1497610002051762303983
            # DF: int_hasCustomer=ss → hasStateIncident=statusIncident__2
            # DF: int_hasCustomer=ss + hasTypeInc=typeIncident__1
            #     → hasSupportGroup=supportGroup_14976691762302662
            # NV: int_hasCustomer=ss → incident_hasOrigin≠incidentOrigin__1
            (
                "Hola quiero completar una query. Tengo el supportCategory_14976212811762302676 y la empresa ss",
                ['ss', 'supportCategory_14976212811762302676',
                 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_14976212821762302676', None]
            ),

            # ─────────────────────────────────────────────────────────────────
            # BLOQUE 5 · Reglas contradictorias  (casos 41–50)
            # Una regla df y una regla noValid se activan sobre el MISMO predicado
            # con valores distintos, generando una contradicción directa.
            # ─────────────────────────────────────────────────────────────────

            # Caso 41 · incident_1497610000541762303983
            # DF: int_hasCustomer=company__3S8A2Y7FV → hasTechnician=employee__429
            # NV: int_hasCustomer=company__3S8A2Y7FV → hasTechnician≠employee__430
            # CONTRADICCIÓN en hasTechnician (df asigna __429, noValid prohíbe __430)
            (
                "Hola quiero completar una query. Tengo el supportCategory_149761931762302662 y la empresa company__3S8A2Y7FV",
                ['company__3S8A2Y7FV', 'supportCategory_149761931762302662',
                 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149761941762302662', 'employee__601']
            ),

            # Caso 42 · incident_1497610000761762303983
            # DF: int_hasCustomer=company__3S8A2Y7FV → hasTechnician=employee__429
            # DF: hasSupportGroup=supportGroup_149761041762302662 + hasTypeInc=typeIncident__2
            #     → hasExternalTechnician=person_1497642801762302663
            # NV: hasSupportGroup=supportGroup_149761041762302662 → hasTechnician≠employee__136
            # NV: int_hasCustomer=company__3S8A2Y7FV → hasTechnician≠employee__430
            # CONTRADICCIÓN en hasTechnician
            (
                "Hola quiero completar una query. Tengo el supportCategory_149761591762302662 y la empresa company__3S8A2Y7FV",
                ['company__3S8A2Y7FV', 'supportCategory_149761591762302662',
                 'typeIncident__2', 'incidentOrigin__2',
                 'supportGroup_149761041762302662', None]
            ),

            # Caso 43 · incident_1497610000911762303983
            # DF: int_hasCustomer=company__3S8A2Y7FV → hasTechnician=employee__429
            # NV: incident_hasOrigin=incidentOrigin__1 → hasTypeInc≠typeIncident__1
            # NV: hasExternalTechnician=person_1497630881762302663 → hasTypeInc≠typeIncident__1
            # NV: int_hasCustomer=company__3S8A2Y7FV → hasTechnician≠employee__430
            # CONTRADICCIÓN en hasTechnician
            (
                "Hola quiero completar una query. Tengo el supportCategory_149761591762302662 y la empresa company__3S8A2Y7FV",
                ['company__3S8A2Y7FV', 'supportCategory_149761591762302662',
                 'typeIncident__2', 'incidentOrigin__1',
                 'supportGroup_149762921762302662', None]
            ),

            # Caso 44 · incident_1497610001081762303983
            # DF: int_hasCustomer=company__3S8A2Y7FV → hasTechnician=employee__429
            # NV: int_hasCustomer=company__3S8A2Y7FV → hasTechnician≠employee__430
            # CONTRADICCIÓN en hasTechnician
            (
                "Hola quiero completar una query. Tengo el supportCategory_149761591762302662 y la empresa company__3S8A2Y7FV",
                ['company__3S8A2Y7FV', 'supportCategory_149761591762302662',
                 'typeIncident__2', 'incidentOrigin__2',
                 'supportGroup_149762921762302662', None]
            ),

            # Caso 45 · incident_1497610001151762303983
            # DF: int_hasCustomer=company__3S8A2Y7FV → hasTechnician=employee__429
            # NV: int_hasCustomer=company__3S8A2Y7FV → hasTechnician≠employee__430
            # NV: hasSupportTeam=supportTeam_149762304211762302899
            #     → hasSupportCategory≠supportCategory_149763514921762303044
            # CONTRADICCIÓN en hasTechnician
            (
                "Hola quiero completar una query. Tengo el supportCategory_149767060481762303533 y la empresa company__3S8A2Y7FV",
                ['company__3S8A2Y7FV', 'supportCategory_149767060481762303533',
                 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149765457361762303308', 'employee__429']
            ),

            # Caso 46 · incident_1497610001171762303983
            # DF: int_hasCustomer=company__3S8A2Y7FV → hasTechnician=employee__429
            # NV: incident_hasOrigin=incidentOrigin__1 → hasTypeInc≠typeIncident__1
            # NV: int_hasCustomer=company__3S8A2Y7FV → hasTechnician≠employee__430
            # CONTRADICCIÓN en hasTechnician
            (
                "Hola quiero completar una query. Tengo el supportCategory_14976411762302662 y la empresa company__3S8A2Y7FV",
                ['company__3S8A2Y7FV', 'supportCategory_14976411762302662',
                 'typeIncident__1', 'incidentOrigin__1',
                 'supportGroup_149762761762302662', 'employee__294']
            ),

            # Caso 47 · incident_1497610001551762303983
            # DF: int_hasCustomer=company__3S8A2Y7FV → hasTechnician=employee__429
            # DF: hasSupportGroup=supportGroup_149761041762302662 + hasTypeInc=typeIncident__2
            #     → hasExternalTechnician=person_1497642801762302663
            # NV: hasSupportGroup=supportGroup_149761041762302662 → hasTechnician≠employee__136
            # NV: int_hasCustomer=company__3S8A2Y7FV → hasTechnician≠employee__430
            # CONTRADICCIÓN en hasTechnician
            (
                "Hola quiero completar una query. Tengo el supportCategory_149761031762302662 y la empresa company__3S8A2Y7FV",
                ['company__3S8A2Y7FV', 'supportCategory_149761031762302662',
                 'typeIncident__2', 'incidentOrigin__2',
                 'supportGroup_149761041762302662', None]
            ),

            # Caso 48 · incident_1497610001631762303983
            # DF: int_hasCustomer=company__3S8A2Y7FV → hasTechnician=employee__429
            # NV: incident_hasOrigin=incidentOrigin__1 → hasTypeInc≠typeIncident__1
            # NV: int_hasCustomer=company__3S8A2Y7FV → hasTechnician≠employee__430
            # CONTRADICCIÓN en hasTechnician
            (
                "Hola quiero completar una query. Tengo el supportCategory_149761591762302662 y la empresa company__3S8A2Y7FV",
                ['company__3S8A2Y7FV', 'supportCategory_149761591762302662',
                 'typeIncident__2', 'incidentOrigin__1',
                 'supportGroup_149762921762302662', 'employee__294']
            ),

            # Caso 49 · incident_1497610001851762303983
            # DF: int_hasCustomer=company__3S8A2Y7FV → hasTechnician=employee__429
            # NV: int_hasCustomer=company__3S8A2Y7FV → hasTechnician≠employee__430
            # CONTRADICCIÓN en hasTechnician
            (
                "Hola quiero completar una query. Tengo el supportCategory_149764691762302662 y la empresa company__3S8A2Y7FV",
                ['company__3S8A2Y7FV', 'supportCategory_149764691762302662',
                 'typeIncident__1', 'incidentOrigin__2',
                 'supportGroup_149762921762302662', None]
            ),

            # Caso 50 · incident_1497610001881762303983
            # DF: int_hasCustomer=company__3S8A2Y7FV → hasTechnician=employee__429
            # NV: int_hasCustomer=company__3S8A2Y7FV → hasTechnician≠employee__430
            # CONTRADICCIÓN en hasTechnician
            (
                "Hola quiero completar una query. Tengo el supportCategory_149765961762302662 y la empresa company__3S8A2Y7FV",
                ['company__3S8A2Y7FV', 'supportCategory_149765961762302662',
                 'typeIncident__2', 'incidentOrigin__2',
                 'supportGroup_149764431762302662', None]
            ),
        ]
        
        
        self.casos_de_prueba = [
            (query, procesar_mensaje_usuario(query)[1])
            for query, _ in self.casos_de_prueba
        ]
        
        print(self.casos_de_prueba)


    def test_sin_reglas(self):
        """Casos 1-10: ninguna regla df ni noValid activa."""
        print("Casos 1-10: ninguna regla df ni noValid activa.")
        
        for i, (query, expected) in enumerate(self.casos_de_prueba[:10], start=1):
            with self.subTest(caso=i, query=query):
                result = process_query(query)
                self.assertEqual(
                    result, expected,
                    msg=f"Caso {i} fallido.\nQuery: {query}\nEsperado: {expected}\nObtenido: {result}"
                )

    def test_reglas_df(self):
        """Casos 11-20: solo reglas df se activan."""
        print("Casos 11-20: solo reglas df se activan.")

        for i, (query, expected) in enumerate(self.casos_de_prueba[10:20], start=11):
            with self.subTest(caso=i, query=query):
                result = process_query(query)
                self.assertEqual(
                    result, expected,
                    msg=f"Caso {i} fallido.\nQuery: {query}\nEsperado: {expected}\nObtenido: {result}"
                )

    def test_reglas_novalid(self):
        """Casos 21-30: solo reglas noValid se activan."""
        print("Casos 21-30: solo reglas noValid se activan.")
        for i, (query, expected) in enumerate(self.casos_de_prueba[20:30], start=21):
            with self.subTest(caso=i, query=query):
                result = process_query(query)
                self.assertEqual(
                    result, expected,
                    msg=f"Caso {i} fallido.\nQuery: {query}\nEsperado: {expected}\nObtenido: {result}"
                )

    def test_ambas_reglas(self):
        """Casos 31-40: reglas df y noValid se activan sin contradicción."""
        print("Casos 31-40: reglas df y noValid se activan sin contradicción.")
        for i, (query, expected) in enumerate(self.casos_de_prueba[30:40], start=31):
            with self.subTest(caso=i, query=query):
                result = process_query(query)
                self.assertEqual(
                    result, expected,
                    msg=f"Caso {i} fallido.\nQuery: {query}\nEsperado: {expected}\nObtenido: {result}"
                )

    def test_reglas_contradictorias(self):
        """Casos 41-50: df y noValid se contradicen sobre el mismo predicado."""
        print("Casos 41-50: df y noValid se contradicen sobre el mismo predicado")
        for i, (query, expected) in enumerate(self.casos_de_prueba[40:50], start=41):
            with self.subTest(caso=i, query=query):
                result = process_query(query)
                self.assertEqual(
                    result, expected,
                    msg=f"Caso {i} fallido.\nQuery: {query}\nEsperado: {expected}\nObtenido: {result}"

        
    #def test_procesamiento_de_queries(self):
    #    for input_text, expected_output in self.casos_de_prueba:
    #        
    #        with self.subTest(input_text=input_text):
    #            resultado = procesar_query(input_text)
    #            self.assertEqual(resultado, expected_output)

# 2. Configuración clave para que no reinicie el kernel de Jupyter:
if __name__ == '__main__':
    unittest.main(argv=[''], verbosity=2, exit=False)
    
    
    print("DF veces")
    print(contadores["vecesdf"])
    print("NO VALUE veces")
    print(contadores["vecesnv"])